# 5-1 Assignment: Cartpole Problem
___
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Run all code blocks and observe the output. Once you have reviewed the sample output. Use the <b>LastName_FirstName_Assignment2.ipynb</b> file to complete your assignment.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> For compatability purposes, libraries have been updated from those used in the required readings to match to current versions; hence some of the package invocations may differ slightly from the book. The affected lines of code have comments added to the right as applicable, with the old code commented out above for reference.
</div>

<div class="alert alert-block alert-danger" style="color:black;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
<b>Example messages:</b>
    <ul>
        <li>Could not find cuda drivers on your machine, GPU will not be used.</li>
        <li>Please check linkage and avoid linking the same target more than once.</li>
        <li>E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)</li>
        <li>Allocation of ######## exceeds 10% of free system memory</li>
    </ul>
</div>

---

### Installing Required Packages
This is to install necessary components to run the assignment

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.1.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If the assignment takes too long, you may want to download the code and run it locally to take advantage of your GPU.
</div>

In [2]:
# Package Imports
import numpy as np
import tensorflow as tf
import gymnasium as gym
from collections import deque
import random

# Environment setup and Variables
try:
    env = gym.make("CartPole-v1", render_mode=None)
except Exception:
    print('Failed to initialize environment! Make sure that gymnasium was installed correctly!')
    sys.exit(1)

state_shape = env.observation_space.shape[0]
action_size = env.action_space.n

# Hyperparameters
GAMMA = 0.99
EXPLORATION_MAX = 1.0
EXPLORATION_MIN = 0.01
EXPLORATION_DECAY = 0.990
LEARNING_RATE = 0.001
BATCH_SIZE = 64
TRAIN_START = 1000
MEMORY_SIZE = 2000
EPISODES = 300

# Counters during training
train_freq = 4
step_count = 0
target_update_freq = 10
# If an error occurs below, it is because the environment is looking for a GPU.

In [3]:
memory = deque(maxlen=MEMORY_SIZE)

# DQN Builder
def build_dqn_model():
    try:
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(state_shape,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='mse')
        return model
    except Exception:
        print('Error occurred while building the DQN model!')
        sys.exit(1)

In [5]:
# Calling the build_dqn_model method
model = build_dqn_model()
target_model = build_dqn_model()
target_model.set_weights(model.get_weights())

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> If an error displays, it is simply trying to connect to the GPU.
</div>


In [6]:
# Random Action selection for exploration
def get_action(state, EXPLORATION_MAX):
    try:
        if np.random.rand() <= EXPLORATION_MAX:
            return random.randrange(action_size)
        q_values = model.predict(np.array([state]), verbose=0) #Remove verbose=0 to see every prediction display
        return np.argmax(q_values[0])
    except Exception:
            print("Error during get_action method!")
            sys.exit(1)

In [7]:
# Experience Replay
def experience_replay():
    if len(memory) < TRAIN_START:
        return

    batch = random.sample(memory, BATCH_SIZE)
    states = np.zeros((BATCH_SIZE, state_shape))
    next_states = np.zeros((BATCH_SIZE, state_shape))
    actions, rewards, completions = [], [], []

    try:
        for i, (state, action, reward, state_next, terminal) in enumerate(batch):
            states[i] = state
            next_states[i] = state_next
            actions.append(action)
            rewards.append(reward)
            completions.append(terminal)

        target_q = model.predict(states, verbose=0) #Remove verbose=0 to see every prediction display
        next_q = target_model.predict(next_states, verbose=0) #Remove verbose=0 to see every prediction display

        for i in range(BATCH_SIZE):
            if completions[i]:
                target_q[i][actions[i]] = rewards[i]
            else:
                target_q[i][actions[i]] = rewards[i] + GAMMA * np.max(next_q[i])

        model.fit(states, target_q, batch_size=BATCH_SIZE, verbose=0)
    except Exception:
        print("Error during replay")
        sys.exit(1)

In [ ]:
# A list to keep track of all of the episodes
episode_rewards = []

# Main loop
# Will iterate through based on the number of EPISODES originally listed up top
for e in range(EPISODES):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = get_action(state, EXPLORATION_MAX)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        reward = reward if not done else -10

        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        step_count += 1
        if step_count % train_freq == 0:
            experience_replay()

    if EXPLORATION_MAX > EXPLORATION_MIN:
        EXPLORATION_MAX *= EXPLORATION_DECAY
        EXPLORATION_MAX = max(EXPLORATION_MIN, EXPLORATION_MAX)

    if e % target_update_freq == 0:
        target_model.set_weights(model.get_weights())

    episode_rewards.append(total_reward)

    # This will display the average reward every 10 episodes
    # Comment this section and add the below print method if you want to display every episode
    if (e + 1) % 10 == 0:
        avg_reward = np.mean(episode_rewards[-10:])
        print(f"Episode {e+1}: Average Reward = {avg_reward:.2f}, Exploration = {EXPLORATION_MAX:.3f}")

    # Logic to complete execution if average is greater than 195
    if len(episode_rewards) >= 100:
        avg_last_hun = np.mean(episode_rewards[-100:])
        if avg_last_hun >= 195:
            print(f"Solved at episode {e+1}: Average reward over last 100 = {avg_last_hun:.2f}")
            break

    # Comment this in if you want to see every iteration of the training occurring
    # print(f"Episode {e+1}: Total Reward = {total_reward}, Exploration = {EXPLORATION_MAX:.3f}")

# Saving model as needed
model.save("Cartpole_model.h5")
print("Training complete! Model saved as Cartpole_model")

I0000 00:00:1775364213.966978     903 service.cc:146] XLA service 0x7fd158003d40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775364213.967034     903 service.cc:154]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775364214.159594     903 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Episode 10: Average Reward = 12.30, Exploration = 0.904
Episode 20: Average Reward = 5.30, Exploration = 0.818
Episode 30: Average Reward = 7.90, Exploration = 0.740
Episode 40: Average Reward = 10.70, Exploration = 0.669
Episode 50: Average Reward = 10.50, Exploration = 0.605
Episode 60: Average Reward = 1.60, Exploration = 0.547
Episode 70: Average Reward = 2.30, Exploration = 0.495
Episode 80: Average Reward = 2.60, Exploration = 0.448
Episode 90: Average Reward = 12.50, Exploration = 0.405
Episode 100: Average Reward = 5.00, Exploration = 0.366
Episode 110: Average Reward = 30.20, Exploration = 0.331
Episode 120: Average Reward = 72.90, Exploration = 0.299
Episode 130: Average Reward = 162.50, Exploration = 0.271
Episode 140: Average Reward = 155.10, Exploration = 0.245


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    If an error displays, it is attempting to connect to the GPU.<br>
    It will not connect and run on CPU. <br>
    Code will take some time to run, which is commonplace for real life models.<br>
    <b><i>-- Make sure your computer does not go to sleep, and take a well-deserved break! --</b></i>
</div>


<div class="alert alert-block alert-success" style="color:black;">
<b>Make your observations here as it pertains to the assignment rubric.</b> 
</div>

In this assignment, reinforcement learning is used to train an agent to balance a pole on a moving cart in the CartPole environment. The goal of the agent is to keep the pole balanced for as long as possible. Every step that the pole stays upright gives the agent a reward, so the agent learns which actions help it stay balanced longer. The state values describe what is happening in the environment at that moment. In the CartPole problem, the state includes the cart position, cart velocity, pole angle, and pole angular velocity. The agent can take two actions, which are moving the cart to the left or to the right.

Experience replay is used to help the model learn more effectively. Instead of only learning from the most recent step, the agent stores past experiences in memory. These experiences include the state, action, reward, next state, and whether the episode ended. During training, the model randomly selects a batch of these past experiences to train the neural network. This helps reduce correlations between steps and makes the learning process more stable.

The neural network in this assignment is used to estimate the Q-values for each possible action. The network takes the current state as input and outputs a predicted value for each action the agent can take. The hidden layers help the model learn patterns in the environment, and the output layer predicts the value of moving left or right. The agent then chooses the action with the highest predicted value.

The discount factor controls how much the agent values future rewards compared to immediate rewards. A higher discount factor means the agent focuses more on long-term rewards. The learning rate controls how quickly the neural network updates during training. If the learning rate is too high, the model may become unstable, but if it is too low, learning may be very slow. Adjusting these parameters can affect how well and how quickly the agent learns to solve the CartPole problem.